In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

# Path to your SQLite DB
db_path = Path("/Users/keerthanavenkatesan/Documents/Data Managment/CS210FinalProject/Spotify_Popularity_Prediction/data/spotify.db")

# Connect
conn = sqlite3.connect(db_path)
print("Connected to database:", db_path)


Connected to database: /Users/keerthanavenkatesan/Documents/Data Managment/CS210FinalProject/Spotify_Popularity_Prediction/data/spotify.db


In [2]:
# 1. Check what tables exist
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Tables in database:\n", tables, "\n")

# 2. Check columns in the tracks table
cols = pd.read_sql("PRAGMA table_info(tracks);", conn)
print("Columns in 'tracks' table:")
display(cols)

# 3. Preview first 5 rows
preview = pd.read_sql("SELECT * FROM tracks LIMIT 5;", conn)
print("\nPreview of tracks table:")
display(preview)


Tables in database:
      name
0  tracks 

Columns in 'tracks' table:


,cid,name,type,notnull,dflt_value,pk
0,0,Unnamed: 0,INTEGER,0,None,0
1,1,artist_name,TEXT,0,None,0
2,2,track_name,TEXT,0,None,0
3,3,track_id,TEXT,0,None,0
4,4,popularity,INTEGER,0,None,0
5,5,year,INTEGER,0,None,0
6,6,genre,TEXT,0,None,0
7,7,danceability,REAL,0,None,0
8,8,energy,REAL,0,None,0
9,9,key,INTEGER,0,None,0



Preview of tracks table:


,Unnamed: 0,artist_name,track_name,track_id,popularity,year,genre,danceability,energy,key,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature,cluster_id
0,1054304,Cabas,Amor De Mis Amores,3FTqjWmL21xi4HTTOq94EQ,38,2006,alt-rock,0.725,0.553,6,...,0,0.0340,0.276000,0.000007,0.1850,0.729,90.009,206653,4,2
1,621408,Ketil Bjørnstad,Første sang,0Pt7ESPgrdTdaxp2f29hX2,11,2023,swedish,0.277,0.164,9,...,0,0.0373,0.878000,0.000181,0.3350,0.184,89.308,459733,4,1
2,1121669,Project 86,Evil (A Chorus Of Resistance),75Ub3ckaoTdzgH9Azeu8cY,38,2007,alt-rock,0.486,0.927,2,...,0,0.0428,0.000003,0.014500,0.0952,0.377,135.540,183373,4,4
3,439351,Ital Tek,Open Heart,5WEPna9GWi0NkqVLAkEKNN,18,2020,dubstep,0.411,0.442,1,...,0,0.0270,0.485000,0.926000,0.1910,0.172,174.019,347610,3,3
4,266036,I-Roy,Irie Right,6peHySxvmZaRF9YEwUsggq,18,2017,dancehall,0.748,0.660,10,...,0,0.2710,0.125000,0.000000,0.0783,0.400,75.583,196179,4,2


In [12]:
query1 = """
SELECT 
    cluster_id,
    COUNT(*) AS count,
    ROUND(AVG(popularity), 2) AS avg_popularity
FROM tracks
GROUP BY cluster_id
ORDER BY avg_popularity DESC;
"""

df_cluster_pop = pd.read_sql(query1, conn)
df_cluster_pop

,cluster_id,count,avg_popularity
0,2,30555,21.63
1,4,24657,19.10
2,5,18293,18.46
3,1,9822,16.81
4,3,14898,13.36
5,0,1775,11.33


In [ ]:
query_genre = """
SELECT 
    cluster_id,
    genre,
    COUNT(*) AS count
FROM tracks
GROUP BY cluster_id, genre
ORDER BY cluster_id, count DESC;
"""

df_cluster_genres = pd.read_sql(query_genre, conn)
df_cluster_genres.head(20)


,cluster_id,genre,count
0,0,comedy,1280
1,0,show-tunes,56
2,0,german,41
3,0,swedish,38
4,0,hip-hop,29
5,0,dancehall,23
6,0,hardcore,19
7,0,sleep,16
8,0,chill,15
9,0,samba,14




This cluster 0 is dominated by comedy tracks (72% of the cluster).
These songs share characteristics such as high speechiness, atypical musical structure, and low mainstream appeal.
As a result, the cluster has the lowest average popularity among all clusters.

In [7]:
df_cluster_genres[df_cluster_genres["cluster_id"] == 1].head(60)

,cluster_id,genre,count
69,1,ambient,1354
70,1,sleep,1298
71,1,new-age,1237
72,1,classical,1182
73,1,guitar,821
74,1,piano,701
75,1,opera,542
76,1,german,353
77,1,jazz,308
78,1,french,205


Cluster 1 represents low-energy, highly acoustic music such as ambient, sleep tracks, classical compositions, piano solos, and new-age genres.
These tracks share properties like soft dynamics, slow tempo, high acousticness, and low danceability.

In [8]:
df_cluster_genres[df_cluster_genres["cluster_id"] == 2].head(60)

,cluster_id,genre,count
137,2,dancehall,1487
138,2,forro,1386
139,2,salsa,1319
140,2,dance,1064
141,2,k-pop,1050
142,2,hip-hop,1040
143,2,disco,904
144,2,pop-film,877
145,2,samba,837
146,2,spanish,781


Cluster 2 contains high-energy, dance-oriented tracks spanning genres such as dancehall, salsa, K-pop, hip-hop, disco, EDM, and funk.
These genres share strong beats, high danceability, energetic profiles, and global popularity.

In [16]:
for cid in [3, 4, 5]:
    print(f"Cluster {cid}")
    display(df_cluster_genres[df_cluster_genres["cluster_id"] == cid].head(30))


Cluster 3


,cluster_id,genre,count
218,3,minimal-techno,1209
219,3,deep-house,851
220,3,drum-and-bass,665
221,3,breakbeat,558
222,3,progressive-house,526
223,3,black-metal,524
224,3,chill,520
225,3,techno,504
226,3,dub,486
227,3,trance,466


Cluster 4


,cluster_id,genre,count
300,4,death-metal,1244
301,4,black-metal,1125
302,4,grindcore,1119
303,4,emo,1093
304,4,heavy-metal,1028
305,4,hardcore,872
306,4,hard-rock,852
307,4,alt-rock,845
308,4,goth,838
309,4,hardstyle,757


Cluster 5


,cluster_id,genre,count
382,5,tango,1096
383,5,acoustic,961
384,5,gospel,827
385,5,cantopop,817
386,5,folk,776
387,5,singer-songwriter,715
388,5,indian,644
389,5,show-tunes,617
390,5,rock-n-roll,600
391,5,opera,599


 cluster 3 is defined by high-intensity electronic genres (minimal-techno, deep-house, DnB, trance, EDM), mixed with experimental or darker tones (black-metal, industrial, trip-hop, goth).
 cluster 4 is overwhelmingly rock, metal, hardcore, punk, goth, and other loud or alternative subgenres.
 cluster 5 represents instrumentals, classic styles, and world genres.

In [14]:
query_top_artists = """
SELECT 
    cluster_id,
    artist_name,
    COUNT(*) AS track_count
FROM tracks
GROUP BY cluster_id, artist_name
HAVING track_count >= 5
ORDER BY cluster_id, track_count DESC;
"""

df_top_artists = pd.read_sql(query_top_artists, conn)
df_top_artists.head(20)


,cluster_id,artist_name,track_count
0,0,Henning Mankell,36
1,0,Jim Gaffigan,22
2,0,Eddie Izzard,19
3,0,Alonzo Bodden,18
4,0,Jim Norton,17
5,0,Die 3 vom Ast,15
6,0,Todd Barry,13
7,0,Steve Hofstetter,13
8,0,Jim Florentine,12
9,0,Dave Waite,12


In [17]:
for cid in [0,1,2,3, 4, 5]:
    print(f" Cluster {cid} ---")
    display(df_top_artists[df_top_artists["cluster_id"] == cid].head(30))


 Cluster 0 ---


,cluster_id,artist_name,track_count
0,0,Henning Mankell,36
1,0,Jim Gaffigan,22
2,0,Eddie Izzard,19
3,0,Alonzo Bodden,18
4,0,Jim Norton,17
5,0,Die 3 vom Ast,15
6,0,Todd Barry,13
7,0,Steve Hofstetter,13
8,0,Jim Florentine,12
9,0,Dave Waite,12


 Cluster 1 ---


,cluster_id,artist_name,track_count
86,1,Traditional,261
87,1,Johann Sebastian Bach,169
88,1,Wolfgang Amadeus Mozart,83
89,1,Ludwig van Beethoven,67
90,1,Little Symphony,66
91,1,Jim Brickman,63
92,1,Andrei Krylov,63
93,1,Hans Zimmer,61
94,1,Steven Halpern,60
95,1,Giacomo Meyerbeer,55


 Cluster 2 ---


...

,cluster_id,artist_name,track_count
1910,3,Grateful Dead,32
1911,3,Armin van Buuren,32
1912,3,Orlando Voorn,30
1913,3,Boris Brejcha,27
1914,3,Lemongrass,26
1915,3,Louie Vega,25
1916,3,Tiësto,24
1917,3,Huda Hudia,23
1918,3,DJ 3000,23
1919,3,Surgeon,22


 Cluster 4 ---


,cluster_id,artist_name,track_count
2535,4,Armin van Buuren,35
2536,4,SICK LEGEND,30
2537,4,Rush,26
2538,4,Agoraphobic Nosebleed,26
2539,4,Sabaton,25
2540,4,Guided By Voices,24
2541,4,Napalm Death,23
2542,4,Ningen Isu,22
2543,4,Nasum,22
2544,4,Iron Maiden,22


 Cluster 5 ---


,cluster_id,artist_name,track_count
3675,5,Grateful Dead,116
3676,5,Traditional,72
3677,5,Elvis Presley,64
3678,5,Francisco Canaro,61
3679,5,Andrew Lloyd Webber,47
3680,5,Carlos Di Sarli,39
3681,5,Willie Nelson,34
3682,5,Denise Gagne,33
3683,5,Aníbal Troilo,33
3684,5,Alfredo De Angelis,33


In [18]:
query_stats = """
SELECT 
    cluster_id,
    ROUND(AVG(danceability), 3) AS avg_danceability,
    ROUND(AVG(energy), 3) AS avg_energy,
    ROUND(AVG(loudness), 3) AS avg_loudness,
    ROUND(AVG(speechiness), 3) AS avg_speechiness,
    ROUND(AVG(acousticness), 3) AS avg_acousticness,
    ROUND(AVG(instrumentalness), 3) AS avg_instrumentalness,
    ROUND(AVG(liveness), 3) AS avg_liveness,
    ROUND(AVG(valence), 3) AS avg_valence,
    ROUND(AVG(tempo), 3) AS avg_tempo
FROM tracks
GROUP BY cluster_id
ORDER BY cluster_id;
"""

df_cluster_stats = pd.read_sql(query_stats, conn)
df_cluster_stats


,cluster_id,avg_danceability,avg_energy,avg_loudness,avg_speechiness,avg_acousticness,avg_instrumentalness,avg_liveness,avg_valence,avg_tempo
0,0,0.574,0.611,-12.802,0.836,0.728,0.022,0.641,0.474,100.780
1,1,0.334,0.166,-21.380,0.050,0.880,0.763,0.153,0.187,103.455
2,2,0.680,0.734,-6.543,0.095,0.213,0.037,0.187,0.703,118.579
3,3,0.614,0.732,-8.774,0.066,0.101,0.774,0.165,0.385,127.105
4,4,0.412,0.851,-5.506,0.098,0.064,0.134,0.320,0.345,135.666
5,5,0.518,0.380,-10.766,0.056,0.689,0.085,0.197,0.399,113.639


In [19]:
query_similar_tracks = """
WITH target AS (
    SELECT 
        danceability AS t_dance,
        energy AS t_energy,
        loudness AS t_loud,
        valence AS t_valence,
        tempo AS t_tempo,
        genre AS t_genre,
        cluster_id AS t_cluster
    FROM tracks
    WHERE track_id = '3FTqjWmL21xi4HTTOq94EQ'
)

SELECT
    tr.track_name,
    tr.artist_name,
    tr.genre,
    tr.popularity,

    -- Manhattan distance combining 5 key audio features
    ABS(tr.danceability - target.t_dance) +
    ABS(tr.energy - target.t_energy) +
    ABS(tr.loudness - target.t_loud) +
    ABS(tr.valence - target.t_valence) +
    ABS(tr.tempo - target.t_tempo) AS similarity_score

FROM tracks tr, target
WHERE tr.track_id != '3FTqjWmL21xi4HTTOq94EQ'
  AND tr.cluster_id = target.t_cluster      -- filter to same cluster
  AND tr.genre = target.t_genre             -- optional tighter filter

ORDER BY similarity_score ASC
LIMIT 10; """

df_similar_tracks = pd.read_sql(query_similar_tracks, conn)
df_similar_tracks

,track_name,artist_name,genre,popularity,similarity_score
0,Let It Go,Tenth Avenue North,alt-rock,27,0.585
1,Oye cantinero,El Tri,alt-rock,44,0.713
2,El Castigo,Los Master Plus,alt-rock,55,1.190
3,Breakdown,Icon For Hire,alt-rock,40,1.381
4,So What,The Mowgli's,alt-rock,38,1.389
5,Listen Out Loud,DREAMERS,alt-rock,35,1.583
6,"Revenge, And a Little More",Unlike Pluto,alt-rock,47,1.711
7,No Other Place,Hollywood Undead,alt-rock,42,1.865
8,World In My Pocket,The Unlikely Candidates,alt-rock,46,2.128
9,Worried Moon,Chris Cornell,alt-rock,39,2.236
